# Automated Regional Impact Auditor (ARIA)
## River Flood Shelter Risk Assessment

**Captain's Log**: Starting analysis of Taiwan's flood risk for emergency shelters using WRA river data and Fire Agency shelter locations.

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import folium
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from urllib.parse import quote
import json

# Load environment variables
load_dotenv()

# Buffer distances from .env
BUFFER_HIGH = int(os.getenv('BUFFER_HIGH', 500))
BUFFER_MED = int(os.getenv('BUFFER_MED', 1000))
BUFFER_LOW = int(os.getenv('BUFFER_LOW', 2000))

print(f"Buffer distances: High={BUFFER_HIGH}m, Medium={BUFFER_MED}m, Low={BUFFER_LOW}m")

## 1. Data Loading & Cleaning

**Captain's Log**: Loading river polygons from WRA API and checking coordinate system.

In [ ]:
# Load river data from WRA
rivers_url = 'https://gic.wra.gov.tw/Gis/gic/API/Google/DownLoad.aspx?fname=RIVERPOLY&filetype=SHP'
rivers = gpd.read_file(rivers_url)

print(f"River data loaded: {len(rivers)} polygons")
print(f"Original CRS: {rivers.crs}")
print(f"Bounds: {rivers.total_bounds}")

# Ensure we're working in EPSG:3826 (Taiwan datum)
if rivers.crs != 'EPSG:3826':
    rivers = rivers.to_crs('EPSG:3826')
    print(f"Converted to CRS: {rivers.crs}")

**Captain's Log**: Loading shelter data from Fire Agency CSV and performing coordinate validation.

In [ ]:
# Load shelter data (you'll need to download this from data.gov.tw)
# Download from: https://data.gov.tw/dataset/73242
# Save as: data/避難收容處所.csv

try:
    shelters_csv = pd.read_csv('data/避難收容處所.csv', encoding='utf-8')
except UnicodeDecodeError:
    shelters_csv = pd.read_csv('data/避難收容處所.csv', encoding='big5')

print(f"Original shelter records: {len(shelters_csv)}")
print(f"Columns: {list(shelters_csv.columns)}")

# Display first few rows to understand the data structure
shelters_csv.head()

In [ ]:
# Data cleaning - remove invalid coordinates
# Assuming coordinate columns are named '經度' and '緯度' (may need adjustment)
lon_col = '經度'  # Adjust based on actual column names
lat_col = '緯度'  # Adjust based on actual column names

# Check if these columns exist
if lon_col not in shelters_csv.columns or lat_col not in shelters_csv.columns:
    print("Available columns:")
    for col in shelters_csv.columns:
        print(f"  - {col}")
    # You may need to adjust column names based on the actual data
else:
    # Filter valid coordinates (Taiwan bounds: lon 119-122, lat 21-26)
    valid_shelters = shelters_csv[
        (shelters_csv[lon_col] >= 119) & (shelters_csv[lon_col] <= 122) &
        (shelters_csv[lat_col] >= 21) & (shelters_csv[lat_col] <= 26) &
        (shelters_csv[lon_col] != 0) & (shelters_csv[lat_col] != 0)
    ].copy()
    
    print(f"Valid shelter records after cleaning: {len(valid_shelters)}")
    print(f"Removed {len(shelters_csv) - len(valid_shelters)} invalid records")
    
    # Convert to GeoDataFrame
    shelters = gpd.GeoDataFrame(
        valid_shelters,
        geometry=gpd.points_from_xy(valid_shelters[lon_col], valid_shelters[lat_col]),
        crs='EPSG:4326'
    )
    
    # Convert to EPSG:3826 for buffer analysis
    shelters = shelters.to_crs('EPSG:3826')
    print(f"Shelters CRS: {shelters.crs}")
    print(f"Shelter bounds: {shelters.total_bounds}")

**Captain's Log**: Loading township boundaries for regional analysis.

In [ ]:
# Load township boundaries
township_url = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/' + quote('鄉(鎮、市、區)界線1140318.zip')
townships = gpd.read_file(township_url)
townships = townships.to_crs('EPSG:3826')

print(f"Township boundaries loaded: {len(townships)} polygons")
print(f"Township CRS: {townships.crs}")
print(f"Available columns: {list(townships.columns)}")

## 2. Multi-Level Buffer Analysis

**Captain's Log**: Creating three-tier buffer zones around rivers for risk assessment.

In [ ]:
# Create multi-level buffer zones
river_buffers_high = rivers.buffer(BUFFER_HIGH)
river_buffers_med = rivers.buffer(BUFFER_MED)
river_buffers_low = rivers.buffer(BUFFER_LOW)

# Convert to GeoDataFrame
buffer_high_gdf = gpd.GeoDataFrame(geometry=river_buffers_high, crs='EPSG:3826')
buffer_med_gdf = gpd.GeoDataFrame(geometry=river_buffers_med, crs='EPSG:3826')
buffer_low_gdf = gpd.GeoDataFrame(geometry=river_buffers_low, crs='EPSG:3826')

# Dissolve to create unified buffer zones
buffer_high_unified = buffer_high_gdf.dissolve()
buffer_med_unified = buffer_med_gdf.dissolve()
buffer_low_unified = buffer_low_gdf.dissolve()

print(f"High risk buffer (500m): {len(buffer_high_unified)} zones")
print(f"Medium risk buffer (1km): {len(buffer_med_unified)} zones")
print(f"Low risk buffer (2km): {len(buffer_low_unified)} zones")

**Captain's Log**: Performing spatial joins to identify shelters in each risk zone.

In [ ]:
# Spatial joins to identify shelters in each buffer zone
shelters_in_high = gpd.sjoin(shelters, buffer_high_unified, how='inner', predicate='intersects')
shelters_in_med = gpd.sjoin(shelters, buffer_med_unified, how='inner', predicate='intersects')
shelters_in_low = gpd.sjoin(shelters, buffer_low_unified, how='inner', predicate='intersects')

print(f"Shelters in high risk zone (500m): {len(shelters_in_high)}")
print(f"Shelters in medium risk zone (1km): {len(shelters_in_med)}")
print(f"Shelters in low risk zone (2km): {len(shelters_in_low)}")

# Assign risk levels (highest priority wins)
# IMPORTANT: Process in order of LOWEST priority to HIGHEST priority
# This ensures higher risk levels override lower ones
shelters['risk_level'] = 'safe'  # Default to safe

# Low risk overrides safe (but will be overridden by medium/high)
shelters.loc[shelters.index.isin(shelters_in_low.index), 'risk_level'] = 'low'
# Medium risk overrides low and safe (but will be overridden by high)
shelters.loc[shelters.index.isin(shelters_in_med.index), 'risk_level'] = 'medium'
# High risk overrides all others (highest priority)
shelters.loc[shelters.index.isin(shelters_in_high.index), 'risk_level'] = 'high'

# Count shelters by risk level
risk_counts = shelters['risk_level'].value_counts()
print("\nShelters by risk level:")
for level, count in risk_counts.items():
    print(f"  {level}: {count}")

# Verify priority logic: check overlaps
print(f"\nOverlap verification:")
print(f"High & Medium overlap: {len(set(shelters_in_high.index) & set(shelters_in_med.index))}")
print(f"High & Low overlap: {len(set(shelters_in_high.index) & set(shelters_in_low.index))}")
print(f"Medium & Low overlap: {len(set(shelters_in_med.index) & set(shelters_in_low.index))}")

## 3. Capacity Gap Analysis

**Captain's Log**: Analyzing shelter capacity gaps by administrative region.

In [ ]:
# Spatial join shelters with townships
shelters_with_township = gpd.sjoin(shelters, townships, how='inner', predicate='intersects')

# Check available capacity column
capacity_col = None
for col in shelters_with_township.columns:
    if '容納人數' in str(col) or 'capacity' in str(col).lower():
        capacity_col = col
        break

print(f"Using capacity column: {capacity_col}")

# Group by township and risk level with capacity analysis
township_analysis = shelters_with_township.groupby(['TOWNNAME', 'risk_level']).agg({
    'geometry': 'count',  # Count of shelters
    capacity_col: 'sum' if capacity_col else lambda x: 0  # Sum capacity
}).rename(columns={'geometry': 'shelter_count', capacity_col: 'total_capacity'}).reset_index()

print("Township risk analysis with capacity (sample):")
print(township_analysis.head(10))

# Create pivot table for better analysis
township_pivot = township_analysis.pivot(index='TOWNNAME', columns='risk_level', 
                                        values=['shelter_count', 'total_capacity']).fillna(0)

# Flatten column names
township_pivot.columns = [f"{col[0]}_{col[1]}" for col in township_pivot.columns]
township_pivot = township_pivot.reset_index()

print("\nTownship pivot analysis (sample):")
print(township_pivot.head(10))

In [ ]:
# Calculate capacity gap analysis for each township
def calculate_capacity_gap(row):
    """Calculate capacity gap metrics for a township"""
    # Safe capacity
    safe_capacity = row.get('total_capacity_safe', 0)
    
    # Risk capacity (high + medium + low)
    risk_capacity = (row.get('total_capacity_high', 0) + 
                    row.get('total_capacity_medium', 0) + 
                    row.get('total_capacity_low', 0))
    
    # Total capacity
    total_capacity = safe_capacity + risk_capacity
    
    # Risk shelter count
    risk_shelters = (row.get('shelter_count_high', 0) + 
                   row.get('shelter_count_medium', 0) + 
                   row.get('shelter_count_low', 0))
    
    # Capacity gap ratio (risk shelters vs safe capacity)
    capacity_gap_ratio = risk_shelters / max(safe_capacity, 1) if safe_capacity > 0 else risk_shelters
    
    # Risk score (weighted by risk level and capacity)
    risk_score = (row.get('shelter_count_high', 0) * 3 + 
                 row.get('shelter_count_medium', 0) * 2 + 
                 row.get('shelter_count_low', 0) * 1) * capacity_gap_ratio
    
    return pd.Series({
        'safe_capacity': safe_capacity,
        'risk_capacity': risk_capacity,
        'total_capacity': total_capacity,
        'risk_shelters': risk_shelters,
        'capacity_gap_ratio': capacity_gap_ratio,
        'risk_score': risk_score
    })

# Apply capacity gap analysis
capacity_analysis = township_pivot.apply(calculate_capacity_gap, axis=1)

# Combine with township names
final_analysis = pd.concat([township_pivot[['TOWNNAME']], capacity_analysis], axis=1)

# Sort by comprehensive risk score (not just high-risk shelter count)
top_10_risk_areas = final_analysis.sort_values('risk_score', ascending=False).head(10)

print("Top 10 High-Risk Areas (Comprehensive Analysis):")
for idx, row in top_10_risk_areas.iterrows():
    print(f"{row['TOWNNAME']}:")
    print(f"  Risk Score: {row['risk_score']:.1f}")
    print(f"  Risk Shelters: {int(row['risk_shelters'])}")
    print(f"  Safe Capacity: {int(row['safe_capacity'])}")
    print(f"  Capacity Gap Ratio: {row['capacity_gap_ratio']:.2f}")
    print()

# Export detailed analysis
top_10_risk_areas.to_csv('top_10_risk_areas.csv', index=False, encoding='utf-8-sig')
print("Top 10 risk analysis exported to 'top_10_risk_areas.csv'")

# Summary statistics
print(f"\nCapacity Gap Analysis Summary:")
print(f"Total townships analyzed: {len(final_analysis)}")
print(f"Average safe capacity per township: {final_analysis['safe_capacity'].mean():.1f}")
print(f"Average risk shelters per township: {final_analysis['risk_shelters'].mean():.1f}")
print(f"Average capacity gap ratio: {final_analysis['capacity_gap_ratio'].mean():.2f}")

## 4. Visualization

**Captain's Log**: Creating interactive risk map and statistical charts.

In [ ]:
# Create interactive map
# Calculate center of the study area
bounds = shelters.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

# Convert back to WGS84 for folium
shelters_wgs84 = shelters.to_crs('EPSG:4326')
rivers_wgs84 = rivers.to_crs('EPSG:4326')
buffer_high_wgs84 = buffer_high_unified.to_crs('EPSG:4326')
buffer_med_wgs84 = buffer_med_unified.to_crs('EPSG:4326')
buffer_low_wgs84 = buffer_low_unified.to_crs('EPSG:4326')
townships_wgs84 = townships.to_crs('EPSG:4326')

# Create base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# Add buffer zones
folium.GeoJson(
    buffer_low_wgs84,
    style_function=lambda x: {'fillColor': 'yellow', 'color': 'none', 'fillOpacity': 0.2},
    name='Low Risk (2km)'
).add_to(m)

folium.GeoJson(
    buffer_med_wgs84,
    style_function=lambda x: {'fillColor': 'orange', 'color': 'none', 'fillOpacity': 0.3},
    name='Medium Risk (1km)'
).add_to(m)

folium.GeoJson(
    buffer_high_wgs84,
    style_function=lambda x: {'fillColor': 'red', 'color': 'none', 'fillOpacity': 0.4},
    name='High Risk (500m)'
).add_to(m)

# Add rivers
folium.GeoJson(
    rivers_wgs84,
    style_function=lambda x: {'color': 'blue', 'weight': 2},
    name='Rivers'
).add_to(m)

# Add shelters with risk-based colors
risk_colors = {
    'high': 'red',
    'medium': 'orange', 
    'low': 'yellow',
    'safe': 'green'
}

for idx, shelter in shelters_wgs84.iterrows():
    risk = shelter['risk_level']
    folium.CircleMarker(
        location=[shelter.geometry.y, shelter.geometry.x],
        radius=5,
        popup=f"Risk: {risk}\nName: {shelter.get('name', 'N/A')}",
        color=risk_colors.get(risk, 'gray'),
        fill=True,
        fillColor=risk_colors.get(risk, 'gray')
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save map
m.save('risk_map_interactive.html')
print("Interactive risk map saved as 'risk_map_interactive.html'")
m

In [ ]:
# Create comprehensive bar chart of top 10 high-risk townships
top_10_chart = top_10_risk_areas.head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Risk Score and Shelter Count
ax1.barh(range(len(top_10_chart)), top_10_chart['risk_score'], 
        color='#FF4444', alpha=0.8, label='Risk Score')
ax1.set_yticks(range(len(top_10_chart)))
ax1.set_yticklabels(top_10_chart['TOWNNAME'], fontsize=10)
ax1.set_xlabel('Comprehensive Risk Score', fontsize=12)
ax1.set_title('Top 10 High-Risk Townships (Risk Score)', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Add shelter count as secondary axis
ax1_twin = ax1.twiny()
ax1_twin.barh(range(len(top_10_chart)), top_10_chart['risk_shelters'], 
              color='#FFA500', alpha=0.6, label='Risk Shelters')
ax1_twin.set_xlabel('Number of Risk Shelters', fontsize=12, color='#FFA500')
ax1_twin.tick_params(axis='x', labelcolor='#FFA500')

# Chart 2: Capacity Analysis
width = 0.35
x = range(len(top_10_chart))
ax2.bar([i - width/2 for i in x], top_10_chart['safe_capacity'], 
        width, label='Safe Capacity', color='#00C851', alpha=0.8)
ax2.bar([i + width/2 for i in x], top_10_chart['risk_capacity'], 
        width, label='Risk Capacity', color='#FF4444', alpha=0.8)

ax2.set_xticks(x)
ax2.set_xticklabels(top_10_chart['TOWNNAME'], rotation=45, ha='right', fontsize=10)
ax2.set_ylabel('Total Capacity', fontsize=12)
ax2.set_title('Capacity Analysis: Safe vs Risk', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('risk_map.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Comprehensive risk analysis chart saved as 'risk_map.png'")

# Display detailed statistics
print(f"\nDetailed Top 10 Analysis:")
print("=" * 80)
for idx, row in top_10_chart.iterrows():
    print(f"{idx+1}. {row['TOWNNAME']}")
    print(f"   Risk Score: {row['risk_score']:.1f}")
    print(f"   Risk Shelters: {int(row['risk_shelters'])} | Safe Capacity: {int(row['safe_capacity'])}")
    print(f"   Capacity Gap Ratio: {row['capacity_gap_ratio']:.2f}")
    print()

## 5. Export Results

**Captain's Log**: Exporting shelter risk audit results.

In [ ]:
# Export shelter risk audit with complete schema compliance
# Identify required columns for the audit
name_col = None
capacity_col = None

# Find name column
for col in shelters.columns:
    if '名稱' in str(col) or 'name' in str(col).lower():
        name_col = col
        break

# Find capacity column
for col in shelters.columns:
    if '容納人數' in str(col) or 'capacity' in str(col).lower():
        capacity_col = col
        break

print(f"Using name column: {name_col}")
print(f"Using capacity column: {capacity_col}")

# Create comprehensive shelter audit with all required fields
shelter_audit = pd.DataFrame()

# Required fields according to assignment specification
shelter_audit['shelter_id'] = range(len(shelters))  # Unique ID
shelter_audit['name'] = shelters[name_col].fillna('Unknown Shelter') if name_col else 'Unknown Shelter'
shelter_audit['risk_level'] = shelters['risk_level']
shelter_audit['capacity'] = shelters[capacity_col].fillna(0) if capacity_col else 0

# Additional useful fields
shelter_audit['longitude'] = shelters.geometry.x
shelter_audit['latitude'] = shelters.geometry.y

# Calculate distance to nearest river for risk shelters
if 'risk_level' in shelters.columns:
    # For risk shelters, calculate approximate distance to river
    risk_shelters = shelters[shelters['risk_level'] != 'safe'].copy()
    if len(risk_shelters) > 0:
        # Simple distance calculation (in meters, since we're in EPSG:3826)
        distances = []
        for idx, shelter in risk_shelters.iterrows():
            # Find nearest river point (simplified)
            min_distance = float('inf')
            for river_idx, river in rivers.iterrows():
                distance = shelter.geometry.distance(river.geometry)
                if distance < min_distance:
                    min_distance = distance
            distances.append(min_distance)
        
        # Add distances to audit
        risk_distances = dict(zip(risk_shelters.index, distances))
        shelter_audit['distance_to_river'] = shelter_audit['shelter_id'].map(
            lambda x: next((v for k, v in risk_distances.items() if k == x), None)
        )
    else:
        shelter_audit['distance_to_river'] = None

# Validate schema compliance
required_fields = ['shelter_id', 'name', 'risk_level', 'capacity']
missing_fields = [field for field in required_fields if field not in shelter_audit.columns]

if missing_fields:
    print(f"WARNING: Missing required fields: {missing_fields}")
else:
    print("✓ All required fields present in shelter audit")

# Display sample of the audit
print(f"\nShelter Risk Audit Sample (first 3 records):")
print(shelter_audit.head(3).to_string(index=False))

# Export to JSON with proper encoding
try:
    shelter_audit.to_json('shelter_risk_audit.json', orient='records', indent=2, force_ascii=False)
    print(f"\n✓ Shelter risk audit exported to 'shelter_risk_audit.json'")
    print(f"✓ Total records: {len(shelter_audit)}")
    print(f"✓ Schema compliance: {len(missing_fields) == 0}")
except Exception as e:
    print(f"✗ Export failed: {e}")

# Summary statistics
print(f"\n=== FINAL SUMMARY ===")
print(f"Total shelters analyzed: {len(shelters)}")
print(f"High risk shelters: {len(shelters[shelters['risk_level'] == 'high'])}")
print(f"Medium risk shelters: {len(shelters[shelters['risk_level'] == 'medium'])}")
print(f"Low risk shelters: {len(shelters[shelters['risk_level'] == 'low'])}")
print(f"Safe shelters: {len(shelters[shelters['risk_level'] == 'safe'])}")

if capacity_col:
    total_capacity = shelters[capacity_col].sum()
    print(f"Total shelter capacity: {total_capacity:,.0f}")
    print(f"Average capacity per shelter: {total_capacity / len(shelters):.1f}")